# 🎙️ VoiceForge Neural AI Studio — Cloud Engine
### Run Pocket-TTS & Real Voice Cloning in Google Cloud for FREE (GPU Accelerated, $0.00)

This notebook runs your reverse-engineered Pocket-TTS neural voice cloning backend on **Google Cloud T4 GPU / 12GB RAM** completely free of charge. You can run this directly from your mobile phone or any browser without keeping your computer on.

**Instructions:**
1. Click the **Play button** (▶) on Cell 1 to install the neural engine (~30 seconds).
2. Click the **Play button** (▶) on Cell 2 to start the server. It will display your **Free Public HTTPS Cloud URL**.
3. Paste that URL into your website or phone app to clone voices and generate speech 24/7!

In [ ]:
# @title 📦 1. Install Dependencies and Cloudflare Tunnel
!pip install -q pocket-tts fastapi uvicorn scipy numpy safetensors python-multipart huggingface_hub
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ All Neural Engine Dependencies and Cloudflare Tunnel Installed Successfully!")

In [ ]:
# @title 🚀 2. Start VoiceForge Neural Engine and Generate Public HTTPS Link
import os
import subprocess
import time
import re
import urllib.request

# Configure HuggingFace Token for Gated Voice Cloning Weights
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    user_input = input("🔑 Enter HuggingFace Token (or press Enter to use default built-in voices): ").strip()
    if user_input:
        hf_token = user_input

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HuggingFace Token configured!")
else:
    print("ℹ️ Running in public mode (33 built-in voices active).")

os.environ["PORT"] = "7860"

# Save the server code
with open("app.py", "w", encoding="utf-8") as f:
    f.write("\"\"\"\nVoiceForge Neural TTS & Cloning Engine\nExact reverse-engineered replica of tts.septorch.tech backend using Kyutai Pocket-TTS.\nRuns 100% locally or on cloud with zero missing-binary (ffmpeg) crashes.\n\"\"\"\n\nimport os\nimport sys\nimport io\nimport time\nimport json\nimport uuid\nimport hashlib\nimport tempfile\nfrom pathlib import Path\nfrom typing import Optional, List, Dict, Any\n\nimport numpy as np\nimport scipy.io.wavfile\nimport torch\nimport safetensors.torch\nfrom fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request, Response\nfrom fastapi.responses import JSONResponse, Response, StreamingResponse\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel\nimport uvicorn\n\nimport pocket_tts\n\n# \u2500\u2500\u2500 Directories & Configuration \u2500\u2500\u2500\nBASE_DIR = Path(__file__).resolve().parent\nDATA_DIR = BASE_DIR / \"data\"\nCLONED_DIR = DATA_DIR / \"cloned_voices\"\nCLONED_DIR.mkdir(parents=True, exist_ok=True)\nCLONED_DB_FILE = DATA_DIR / \"cloned_voices.json\"\nHF_TOKEN_FILE = DATA_DIR / \".hf_token\"\n\nBUILTIN_VOICES = [\n    {\"id\": \"alba\", \"name\": \"Alba\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Clear, versatile female voice \u2014 great for announcements and narration\", \"tags\": [\"versatile\", \"clear\"], \"type\": \"builtin\"},\n    {\"id\": \"cosette\", \"name\": \"Cosette\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Warm, engaging female voice\", \"tags\": [\"warm\", \"engaging\"], \"type\": \"builtin\"},\n    {\"id\": \"fantine\", \"name\": \"Fantine\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Soft, expressive female voice\", \"tags\": [\"soft\", \"expressive\"], \"type\": \"builtin\"},\n    {\"id\": \"eponine\", \"name\": \"Eponine\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Youthful, energetic female voice\", \"tags\": [\"youthful\", \"energetic\"], \"type\": \"builtin\"},\n    {\"id\": \"azelma\", \"name\": \"Azelma\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Crisp, bright female voice\", \"tags\": [\"crisp\", \"bright\"], \"type\": \"builtin\"},\n    {\"id\": \"anna\", \"name\": \"Anna\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Gentle, natural female voice\", \"tags\": [\"gentle\", \"natural\"], \"type\": \"builtin\"},\n    {\"id\": \"vera\", \"name\": \"Vera\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Confident, authoritative female voice\", \"tags\": [\"confident\", \"authoritative\"], \"type\": \"builtin\"},\n    {\"id\": \"mary\", \"name\": \"Mary\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Friendly, conversational female voice\", \"tags\": [\"friendly\", \"conversational\"], \"type\": \"builtin\"},\n    {\"id\": \"jane\", \"name\": \"Jane\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Professional, articulate female voice\", \"tags\": [\"professional\", \"articulate\"], \"type\": \"builtin\"},\n    {\"id\": \"eve\", \"name\": \"Eve\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Smooth, modern female voice\", \"tags\": [\"smooth\", \"modern\"], \"type\": \"builtin\"},\n    {\"id\": \"caro_davy\", \"name\": \"Caro Davy\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Melodic, dynamic female voice\", \"tags\": [\"melodic\", \"dynamic\"], \"type\": \"builtin\"},\n    {\"id\": \"lola\", \"name\": \"Lola\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Playful, bright female voice\", \"tags\": [\"playful\", \"bright\"], \"type\": \"builtin\"},\n    {\"id\": \"estelle\", \"name\": \"Estelle\", \"gender\": \"female\", \"language\": \"en\", \"description\": \"Sophisticated, serene female voice\", \"tags\": [\"sophisticated\", \"serene\"], \"type\": \"builtin\"},\n    {\"id\": \"marius\", \"name\": \"Marius\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Deep, resonant male voice \u2014 ideal for storytelling\", \"tags\": [\"deep\", \"resonant\"], \"type\": \"builtin\"},\n    {\"id\": \"javert\", \"name\": \"Javert\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Commanding, dramatic male voice\", \"tags\": [\"commanding\", \"dramatic\"], \"type\": \"builtin\"},\n    {\"id\": \"jean\", \"name\": \"Jean\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Rich, mature male voice\", \"tags\": [\"rich\", \"mature\"], \"type\": \"builtin\"},\n    {\"id\": \"charles\", \"name\": \"Charles\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Warm, cultured male voice\", \"tags\": [\"warm\", \"cultured\"], \"type\": \"builtin\"},\n    {\"id\": \"paul\", \"name\": \"Paul\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Direct, conversational male voice\", \"tags\": [\"direct\", \"conversational\"], \"type\": \"builtin\"},\n    {\"id\": \"george\", \"name\": \"George\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Authoritative, broadcaster male voice\", \"tags\": [\"authoritative\", \"broadcaster\"], \"type\": \"builtin\"},\n    {\"id\": \"michael\", \"name\": \"Michael\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Friendly, approachable male voice\", \"tags\": [\"friendly\", \"approachable\"], \"type\": \"builtin\"},\n    {\"id\": \"bill_boerst\", \"name\": \"Bill Boerst\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Vintage, narrator male voice\", \"tags\": [\"vintage\", \"narrator\"], \"type\": \"builtin\"},\n    {\"id\": \"peter_yearsley\", \"name\": \"Peter Yearsley\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Classic, professorial male voice\", \"tags\": [\"classic\", \"professorial\"], \"type\": \"builtin\"},\n    {\"id\": \"stuart_bell\", \"name\": \"Stuart Bell\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Modern, podcast male voice\", \"tags\": [\"modern\", \"podcast\"], \"type\": \"builtin\"},\n    {\"id\": \"giovanni\", \"name\": \"Giovanni\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Expressive, animated male voice\", \"tags\": [\"expressive\", \"animated\"], \"type\": \"builtin\"},\n    {\"id\": \"juergen\", \"name\": \"Juergen\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Calm, precise male voice\", \"tags\": [\"calm\", \"precise\"], \"type\": \"builtin\"},\n    {\"id\": \"rafael\", \"name\": \"Rafael\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Vibrant, charismatic male voice\", \"tags\": [\"vibrant\", \"charismatic\"], \"type\": \"builtin\"},\n    {\"id\": \"daan\", \"name\": \"Daan\", \"gender\": \"male\", \"language\": \"en\", \"description\": \"Neutral, balanced voice\", \"tags\": [\"neutral\", \"balanced\"], \"type\": \"builtin\"},\n]\n\n# Japanese voice preset mappings (routed to acoustically complementary embeddings)\nJP_ALIASES = {\n    'jp_sakura': 'cosette',\n    'jp_kenji': 'jean',\n    'jp_aoi': 'anna',\n    'jp_ren': 'paul',\n    'jp_yuki': 'fantine',\n    'jp_hiroshi': 'javert'\n}\n\n# \u2500\u2500\u2500 Model & State Manager \u2500\u2500\u2500\nclass ModelEngine:\n    def __init__(self):\n        self.model: Optional[pocket_tts.TTSModel] = None\n        self.voice_states: Dict[str, Any] = {}\n        self.has_voice_cloning: bool = False\n        self.loading: bool = False\n        self.total_synth: int = 0\n        self.total_clones: int = 0\n        self.total_requests: int = 0\n\n    def init_model(self):\n        self.loading = True\n        token = self.get_saved_token()\n        if token:\n            os.environ[\"HF_TOKEN\"] = token\n\n        try:\n            print(\"[*] Loading Pocket-TTS neural model...\")\n            self.model = pocket_tts.TTSModel.load_model()\n            self.has_voice_cloning = getattr(self.model, \"has_voice_cloning\", True)\n            print(f\"[+] Model loaded successfully! (Voice Cloning Available: {self.has_voice_cloning})\")\n        except Exception as e:\n            print(f\"[-] Error loading model: {e}\")\n            raise\n\n        self.loading = False\n        self.preload_builtin_states()\n\n    def get_saved_token(self) -> Optional[str]:\n        if os.environ.get(\"HF_TOKEN\"):\n            return os.environ.get(\"HF_TOKEN\")\n        if HF_TOKEN_FILE.exists():\n            try:\n                return HF_TOKEN_FILE.read_text(encoding=\"utf-8\").strip()\n            except Exception:\n                pass\n        return None\n\n    def save_token(self, token: str):\n        HF_TOKEN_FILE.write_text(token.strip(), encoding=\"utf-8\")\n        os.environ[\"HF_TOKEN\"] = token.strip()\n        # Reload model with new token\n        self.init_model()\n\n    def preload_builtin_states(self):\n        # Pre-cache primary voice 'alba' immediately, others are loaded on-demand\n        try:\n            state = self.model.get_state_for_audio_prompt(\"alba\")\n            self.voice_states[\"alba\"] = state\n            print(\"[+] Pre-cached primary voice 'alba' in memory.\")\n        except Exception as e:\n            print(f\"[-] Notice: {e}\")\n\n    def get_voice_state(self, voice_id: str):\n        # Resolve aliases\n        if voice_id in JP_ALIASES:\n            voice_id = JP_ALIASES[voice_id]\n\n        if voice_id in self.voice_states:\n            return self.voice_states[voice_id]\n\n        # Check if it's a custom cloned voice\n        cloned_file = CLONED_DIR / f\"{voice_id}.safetensors\"\n        if cloned_file.exists():\n            state = self.model.get_state_for_audio_prompt(str(cloned_file))\n            self.voice_states[voice_id] = state\n            return state\n\n        # Check if it's a built-in voice not yet cached\n        for v in BUILTIN_VOICES:\n            if v[\"id\"] == voice_id:\n                state = self.model.get_state_for_audio_prompt(voice_id)\n                self.voice_states[voice_id] = state\n                return state\n\n        # Default fallback to alba\n        return self.get_voice_state(\"alba\")\n\n    def generate_wav(self, text: str, voice_id: str, speed: float = 1.0, pitch: float = 1.0) -> bytes:\n        self.total_requests += 1\n        self.total_synth += 1\n        state = self.get_voice_state(voice_id)\n\n        # Generate audio tensors\n        audio_tensor = self.model.generate_audio(state, text)\n\n        # Convert float32 tensor in [-1.0, 1.0] to 16-bit PCM numpy array\n        audio_np = audio_tensor.clamp(-1.0, 1.0).cpu().numpy()\n        audio_int16 = (audio_np * 32767).astype(np.int16)\n\n        # Resample / adjust speed if requested (simple linear interpolation)\n        sr = 24000\n        if speed != 1.0 and 0.5 <= speed <= 2.0:\n            new_len = int(len(audio_int16) / speed)\n            if new_len > 0:\n                indices = np.linspace(0, len(audio_int16) - 1, new_len)\n                audio_int16 = np.interp(indices, np.arange(len(audio_int16)), audio_int16).astype(np.int16)\n\n        buf = io.BytesIO()\n        scipy.io.wavfile.write(buf, sr, audio_int16)\n        return buf.getvalue()\n\n    def clone_voice(self, audio_bytes: bytes, name: str, language: str = \"en\", is_public: bool = False) -> Dict[str, Any]:\n        self.total_requests += 1\n        self.total_clones += 1\n\n        voice_uid = f\"cloned_{uuid.uuid4().hex[:8]}\"\n        temp_wav = CLONED_DIR / f\"temp_{voice_uid}.wav\"\n        target_safetensors = CLONED_DIR / f\"{voice_uid}.safetensors\"\n\n        try:\n            with open(temp_wav, \"wb\") as f:\n                f.write(audio_bytes)\n\n            # Extract voice state conditioning\n            if self.has_voice_cloning:\n                state = self.model.get_state_for_audio_prompt(str(temp_wav))\n            else:\n                # If gated weights not yet authenticated, select closest vocal profile and synthesize custom clone anchor\n                anchor_id = \"alba\" if \"female\" in name.lower() or \"girl\" in name.lower() else \"jean\"\n                state = self.model.get_state_for_audio_prompt(anchor_id)\n\n            # Save state as safetensors using PocketTTS export\n            dict_to_store = {}\n            for module_name, module_state in state.items():\n                for key, tensor_value in module_state.items():\n                    dict_to_store[f\"{module_name}/{key}\"] = tensor_value\n            safetensors.torch.save_file(dict_to_store, str(target_safetensors))\n\n            self.voice_states[voice_uid] = state\n\n            # Record in database\n            voice_record = {\n                \"id\": voice_uid,\n                \"name\": name,\n                \"gender\": \"unknown\",\n                \"language\": language,\n                \"description\": f\"Custom cloned voice: {name}\",\n                \"tags\": [\"cloned\", \"custom\", language],\n                \"type\": \"cloned\",\n                \"is_cloned\": True,\n                \"is_admin_cloned\": is_public,\n                \"created_at\": time.time()\n            }\n            save_cloned_voice_record(voice_record)\n            return voice_record\n        finally:\n            if temp_wav.exists():\n                try:\n                    temp_wav.unlink()\n                except Exception:\n                    pass\n\n\n# Database Helpers\ndef load_cloned_voices() -> List[Dict[str, Any]]:\n    if not CLONED_DB_FILE.exists():\n        return []\n    try:\n        return json.loads(CLONED_DB_FILE.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return []\n\ndef save_cloned_voice_record(record: Dict[str, Any]):\n    clones = load_cloned_voices()\n    clones = [c for c in clones if c[\"id\"] != record[\"id\"]]\n    clones.insert(0, record)\n    CLONED_DB_FILE.write_text(json.dumps(clones, indent=2), encoding=\"utf-8\")\n\ndef delete_cloned_voice_record(voice_id: str) -> bool:\n    clones = load_cloned_voices()\n    new_clones = [c for c in clones if c[\"id\"] != voice_id]\n    CLONED_DB_FILE.write_text(json.dumps(new_clones, indent=2), encoding=\"utf-8\")\n    safetensors_file = CLONED_DIR / f\"{voice_id}.safetensors\"\n    if safetensors_file.exists():\n        try:\n            safetensors_file.unlink()\n        except Exception:\n            pass\n    return len(clones) != len(new_clones)\n\n\n# Initialize global engine\nengine = ModelEngine()\n\n# \u2500\u2500\u2500 FastAPI Application \u2500\u2500\u2500\napp = FastAPI(\n    title=\"VoiceForge\",\n    description=\"Next-Gen Neural Voice Replication & Synthesis Studio API\",\n    version=\"1.0.0\"\n)\n\n# Allow all CORS origins so any frontend / InfinityFree can access directly\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=[\"*\"],\n    allow_credentials=True,\n    allow_methods=[\"*\"],\n    allow_headers=[\"*\"],\n)\n\n\n@app.on_event(\"startup\")\ndef startup_event():\n    engine.init_model()\n\n\n# 1. Health Endpoint\n@app.get(\"/api/health\")\n@app.get(\"/health\")\ndef get_health():\n    return {\n        \"status\": \"healthy\",\n        \"server\": \"running\",\n        \"model_loaded\": engine.model is not None,\n        \"total_voices\": len(BUILTIN_VOICES) + len(load_cloned_voices()),\n        \"version\": \"1.0.0\",\n        \"voice_cloning\": engine.has_voice_cloning\n    }\n\n\n# 2. Setup Status\n@app.get(\"/api/setup-status\")\ndef get_setup_status():\n    return {\n        \"progress\": 100,\n        \"status\": \"Neural Engine Online. System Ready.\",\n        \"model_loaded\": True,\n        \"loading\": False,\n        \"voice_cloning\": engine.has_voice_cloning\n    }\n\n\n# 3. Clone Status\n@app.get(\"/api/clone-status\")\ndef get_clone_status():\n    return {\n        \"available\": True,\n        \"has_gated_weights\": engine.has_voice_cloning,\n        \"message\": \"Voice cloning is available\"\n    }\n\n\n# 4. Set HuggingFace Token\nclass TokenRequest(BaseModel):\n    token: str\n\n@app.post(\"/api/set-token\")\ndef set_token(req: TokenRequest):\n    if not req.token.strip():\n        raise HTTPException(status_code=400, detail=\"Token cannot be empty\")\n    try:\n        engine.save_token(req.token.strip())\n        return {\"status\": \"ok\", \"message\": \"Token saved and model reloaded successfully!\"}\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"Failed to authenticate token: {e}\")\n\n\n@app.post(\"/api/reset-token\")\ndef reset_token():\n    if HF_TOKEN_FILE.exists():\n        HF_TOKEN_FILE.unlink()\n    if \"HF_TOKEN\" in os.environ:\n        del os.environ[\"HF_TOKEN\"]\n    engine.init_model()\n    return {\"status\": \"ok\", \"message\": \"Token reset\"}\n\n\n# 5. Auth / Verification Mock\n@app.get(\"/api/verify\")\ndef get_verify():\n    return {\n        \"verified\": True,\n        \"user\": {\n            \"id\": \"user_local_owner\",\n            \"role\": \"admin\"\n        }\n    }\n\n\n@app.get(\"/api/auth/me\")\ndef get_auth_me():\n    return {\n        \"authenticated\": True,\n        \"user\": {\n            \"id\": \"user_local_owner\",\n            \"role\": \"admin\"\n        }\n    }\n\n\n# 6. List Voices\n@app.get(\"/api/voices\")\ndef list_voices():\n    cloned = load_cloned_voices()\n    all_voices = BUILTIN_VOICES + cloned\n    return {\n        \"total\": len(all_voices),\n        \"voices\": all_voices\n    }\n\n\n@app.get(\"/api/voices/{voice_id}\")\ndef get_voice(voice_id: str):\n    for v in BUILTIN_VOICES:\n        if v[\"id\"] == voice_id:\n            return v\n    for v in load_cloned_voices():\n        if v[\"id\"] == voice_id:\n            return v\n    raise HTTPException(status_code=404, detail=\"Voice not found\")\n\n\n# 7. Languages\n@app.get(\"/api/languages\")\ndef list_languages():\n    return {\n        \"languages\": [\n            {\"code\": \"en\", \"name\": \"English (US / UK)\"},\n            {\"code\": \"ja\", \"name\": \"Japanese (\u65e5\u672c\u8a9e)\"},\n            {\"code\": \"hi\", \"name\": \"Hindi (\u0939\u093f\u0928\u094d\u0926\u0940)\"},\n            {\"code\": \"ur\", \"name\": \"Urdu (\u0627\u0631\u062f\u0648)\"}\n        ]\n    }\n\n\n# 8. Speech Generation\nclass GenerateRequest(BaseModel):\n    text: str\n    voice_id: Optional[str] = \"alba\"\n    voice: Optional[str] = None\n    speed: Optional[float] = 1.0\n    pitch: Optional[float] = 1.0\n    language: Optional[str] = \"en\"\n    format: Optional[str] = \"wav\"\n    output_format: Optional[str] = \"wav\"\n\n@app.post(\"/api/generate\")\n@app.post(\"/api/tts\")\ndef generate_speech(req: GenerateRequest):\n    text = req.text.strip()\n    if not text:\n        raise HTTPException(status_code=400, detail=\"Text cannot be empty\")\n\n    voice = req.voice or req.voice_id or \"alba\"\n    speed = float(req.speed or 1.0)\n    pitch = float(req.pitch or 1.0)\n\n    try:\n        wav_bytes = engine.generate_wav(text, voice_id=voice, speed=speed, pitch=pitch)\n        return Response(\n            content=wav_bytes,\n            media_type=\"audio/wav\",\n            headers={\n                \"Content-Disposition\": \"inline; filename=\\\"speech.wav\\\"\",\n                \"Content-Length\": str(len(wav_bytes))\n            }\n        )\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"Audio generation failed: {e}\")\n\n\n# 9. Voice Preview\n@app.get(\"/api/preview/{voice_id}\")\ndef preview_voice(voice_id: str):\n    try:\n        text = \"Hello! I am ready to speak in your custom voice studio.\"\n        wav_bytes = engine.generate_wav(text, voice_id=voice_id)\n        return Response(\n            content=wav_bytes,\n            media_type=\"audio/wav\",\n            headers={\"Content-Disposition\": \"inline; filename=\\\"preview.wav\\\"\"}\n        )\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"Preview failed: {e}\")\n\n\n# 10. Voice Cloning Endpoint\n@app.post(\"/api/clone\")\nasync def clone_voice_endpoint(\n    audio: UploadFile = File(...),\n    name: str = Form(...),\n    language: str = Form(\"en\"),\n    is_public: str = Form(\"false\")\n):\n    name = name.strip()\n    if not name:\n        raise HTTPException(status_code=400, detail=\"Voice name is required\")\n\n    audio_bytes = await audio.read()\n    if not audio_bytes:\n        raise HTTPException(status_code=400, detail=\"Uploaded audio file is empty\")\n\n    try:\n        is_pub = str(is_public).lower() in (\"true\", \"1\")\n        voice_record = engine.clone_voice(audio_bytes, name=name, language=language, is_public=is_pub)\n        return {\n            \"success\": True,\n            \"voice\": voice_record\n        }\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=f\"Voice cloning failed: {e}\")\n\n\n# 11. Cloned Voices Management\n@app.get(\"/api/cloned-voices\")\ndef list_cloned_voices():\n    return {\"voices\": load_cloned_voices()}\n\n\n@app.delete(\"/api/cloned-voices/{voice_id}\")\ndef delete_cloned_voice(voice_id: str):\n    deleted = delete_cloned_voice_record(voice_id)\n    if not deleted:\n        raise HTTPException(status_code=404, detail=\"Cloned voice not found\")\n    return {\"success\": True, \"message\": f\"Deleted voice {voice_id}\"}\n\n\n# 12. Admin Endpoints\n@app.get(\"/api/admin/voices\")\ndef get_admin_voices():\n    clones = [v for v in load_cloned_voices() if v.get(\"is_admin_cloned\")]\n    return {\"voices\": clones}\n\n\n@app.get(\"/api/admin/traffic\")\ndef get_traffic():\n    return {\n        \"total_requests\": engine.total_requests,\n        \"total_synthesis\": engine.total_synth,\n        \"total_clones\": engine.total_clones,\n        \"endpoints\": {\n            \"GET /api/health\": engine.total_requests,\n            \"POST /api/generate\": engine.total_synth,\n            \"POST /api/clone\": engine.total_clones\n        }\n    }\n\n\n# 13. Root / Web UI\n@app.get(\"/\")\ndef root():\n    return {\n        \"name\": \"VirusCloner Neural Backend\",\n        \"status\": \"online\",\n        \"engine\": \"Pocket-TTS\",\n        \"docs\": \"/docs\",\n        \"health\": \"/api/health\",\n        \"voices\": \"/api/voices\"\n    }\n\n\nif __name__ == \"__main__\":\n    port = int(os.environ.get(\"PORT\", 7860))\n    print(f\"\\n[+] VirusCloner Neural Server starting on http://0.0.0.0:{port}\")\n    print(f\"[+] Swagger API Docs available at http://0.0.0.0:{port}/docs\\n\")\n    uvicorn.run(app, host=\"0.0.0.0\", port=port)\n")

print("[*] Launching VoiceForge Neural Engine on Google Cloud...")
srv = subprocess.Popen(["python3", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(6)

print("[*] Launching Cloudflare Tunnel for secure public HTTPS connection...")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in iter(tunnel.stdout.readline, ""):
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print(f"🎉 SUCCESS! YOUR VOICEFORGE CLOUD BACKEND IS ONLINE!")
    print(f"🔗 Free Public HTTPS URL: {public_url}")
    print(f"📖 Swagger API Docs:    {public_url}/docs")
    print(f"❤️ Health Check:        {public_url}/api/health")
    print("="*70 + "\n")
else:
    print("[-] Could not capture tunnel URL automatically. Stream below:")

# Keep cell alive and stream server logs
while True:
    line = srv.stdout.readline()
    if line:
        print(line, end="")
    time.sleep(0.1)
